# SSFL on Google Colab

Runs the Semi-Supervised Federated Learning framework on a Colab GPU.

**Before you start:** `Runtime → Change runtime type → GPU (T4)`.

Steps: check GPU → get the code → install → run tests → quick sanity run → real experiments → plot results.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Get the code

Pick **one** option:
* **A — GitHub** (default): clones the public repository below — always the latest pushed code.
* **B — upload a zip**: set `GITHUB_URL = ""` and choose `ssfl_framework.zip`.
  For a private fork, set `GITHUB_URL` to it. For a **private** repo, add a
  GitHub personal access token (read access to the repo's contents) in Colab's 🔑 *Secrets* panel under the name
  `GITHUB_TOKEN` and enable notebook access; it is never printed or saved in the notebook.

Optionally set `USE_DRIVE = True` to keep datasets and results in Google Drive
(they are lost when the Colab runtime resets otherwise).

In [ ]:
GITHUB_URL = "https://github.com/ozanyildirim22/FederatedFramework.git"   # set to "" to upload a zip instead
USE_DRIVE  = False   # True = store data/ and runs/ in Google Drive (MyDrive/ssfl)

import os, shutil, zipfile
os.chdir("/content")
if GITHUB_URL:
    shutil.rmtree("/content/FederatedFramework", ignore_errors=True)
    url = GITHUB_URL
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        url = GITHUB_URL.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass                                        # public repo / no secret set
    rc = os.system(f"git clone -q {url} /content/FederatedFramework")
    assert rc == 0, "git clone failed - check GITHUB_URL (and GITHUB_TOKEN for private repos)"
else:
    from google.colab import files
    uploaded = files.upload()                       # choose ssfl_framework.zip
    zip_name = next(iter(uploaded))
    shutil.rmtree("/content/FederatedFramework", ignore_errors=True)
    zipfile.ZipFile(zip_name).extractall("/content/FederatedFramework")
os.chdir("/content/FederatedFramework")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = "/content/drive/MyDrive/ssfl"
else:
    STORE = "/content/ssfl_store"
DATA_DIR, RUNS_DIR = f"{STORE}/data", f"{STORE}/runs"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)
print(os.listdir("."), "| data ->", DATA_DIR, "| runs ->", RUNS_DIR)

## 2. Install (PyTorch is already on Colab)

In [ ]:
!pip install -q -e ".[gui]" medmnist pytest
!python -m ssfl --list

## Optional: SSFL Studio (graphical interface)
Starts the GUI and prints a public **`https://….gradio.live`** link — open it in a new browser tab.
* The GUI runs as a **fresh process**, so it always uses the code cloned above (no stale imports).
  Check the **build** chip in the GUI header: it shows the git commit it is running.
* The download and output folders are pre-filled with `DATA_DIR` and `RUNS_DIR`, also after you
  click a scenario card. DataLoader workers are chosen automatically (shown in the Log tab).
* On a large GPU (A100 / H100 / RTX PRO) tick **⚡ Fast GPU mode** in *Key settings*:
  augmentation runs on the GPU and training uses bf16.
* Training runs on Colab's GPU. **Stop this cell** to close the GUI (training stops too).

In [ ]:
# Fresh process = always the latest cloned code. The public link appears below after ~20 s.
!python -m ssfl --gui --share --data-dir "{DATA_DIR}" --runs-dir "{RUNS_DIR}"

## 3. Unit tests (synthetic data, ~1 min)

In [ ]:
!python -m pytest -q tests

## 4. Quick sanity run (synthetic data, seconds)

In [ ]:
!python -m ssfl -c configs/quick_test.yaml --set output_dir={RUNS_DIR}

## 5. Real experiments

`COMMON` applies to every run. On a large GPU set `FAST = True` (GPU augmentation + bf16). DataLoader workers are chosen automatically: on Colab, background
workers do the augmentation in parallel with the GPU for the real datasets.
Lower `ROUNDS` for a quick check; the thesis settings are 200 (CIFAR-10) and 100 (MedMNIST).
Each run prints one line per round and writes `metrics.csv` to `RUNS_DIR`.

In [ ]:
ROUNDS = 20
FAST = False    # True on A100 / H100 / RTX PRO: augmentation on the GPU + bf16 precision
COMMON = f"data.root={DATA_DIR} output_dir={RUNS_DIR} server.rounds={ROUNDS}"
if FAST:
    COMMON += " data.augment_device=gpu strategy.precision=bf16"

def run(config, name, overrides=""):
    cmd = f"python -m ssfl -c {config} --set name={name} {COMMON} {overrides}"
    print("$", cmd)
    get_ipython().system(cmd)

### 5a. OrganAMNIST — cross-silo (10 clients, C = 1.0), 5% labels

In [ ]:
run("configs/organamnist_fedlabel.yaml", "organ_fedlabel_5", "data.labeled_ratio=0.05")

### 5b. Algorithm comparison on OrganAMNIST
Same data split (same seed) for every algorithm. `strategy.params={}` resets the preset's FedLabel
parameters so each algorithm starts from its own defaults (see `configs/grids/thesis.yaml` for tuned values).
* `fedavg`, `fedprox`: labeled data only (baselines)
* `fedmatch`: inter-client consistency + sigma/psi decomposition
* `fedtrinet`: local / global / combined models, dynamic threshold (FedAvg pre-training for `pretrain_rounds`)
* `fedlabel`: local-vs-global selective pseudo-labeling

In [ ]:
for algo in ["fedavg", "fedprox", "fedmatch", "fedtrinet", "fedlabel"]:
    params = "" if algo == "fedlabel" else "strategy.params={}"
    if algo == "fedtrinet":
        params += f" strategy.params.pretrain_rounds={max(1, ROUNDS // 5)}"
    run("configs/organamnist_fedlabel.yaml", f"organ_{algo}_5", f"strategy.name={algo} data.labeled_ratio=0.05 {params}")

### 5c. Full thesis grid (resumable)
33 runs: CIFAR-10 (20%, 50%), OrganAMNIST and PathMNIST (5%, 20%) x FedAvg, FedProx, FedMatch, FedTriNet,
FedLabel, plus the FedLabel ablations. Finished runs are skipped, so with `USE_DRIVE = True` you can
re-run this cell after a Colab timeout and it continues. `--only` filters by name, `--dry-run` lists the runs.

In [ ]:
GRID_COMMON = f"data.root={DATA_DIR} output_dir={RUNS_DIR}"
!python -m ssfl --grid configs/grids/thesis.yaml --dry-run --set {GRID_COMMON}
# !python -m ssfl --grid configs/grids/thesis.yaml --only organamnist --set {GRID_COMMON}
# !python -m ssfl --grid configs/grids/thesis.yaml --only pathmnist --set {GRID_COMMON}
# !python -m ssfl --grid configs/grids/thesis.yaml --only cifar10 --set {GRID_COMMON}

## 6. Plot results

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

FILTER = ""      # e.g. "organamnist" or "l0p05": only plot runs whose name contains this text
runs = sorted(p for p in glob.glob(f"{RUNS_DIR}/*/metrics.csv") if FILTER in p)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
summary = []
for path in runs:
    df = pd.read_csv(path)
    name = os.path.basename(os.path.dirname(path)).rsplit("_", 1)[0]
    if "test_acc" in df:
        d = df.dropna(subset=["test_acc"])
        axes[0].plot(d["round"], d["test_acc"], label=name)
        summary.append({"run": name, "best_acc": d["test_acc"].max(), "final_acc": d["test_acc"].iloc[-1]})
    if "train_mask_rate" in df:
        axes[1].plot(df["round"], df["train_mask_rate"], label=name)
    if "train_pl_acc" in df:
        axes[2].plot(df["round"], df["train_pl_acc"], label=name)
for ax, t in zip(axes, ["Test accuracy", "Pseudo-label mask rate", "Pseudo-label accuracy"]):
    ax.set_title(t); ax.set_xlabel("round"); ax.grid(alpha=.3)
if runs:   # one legend, outside the plots, so many runs still fit
    axes[2].legend(*axes[0].get_legend_handles_labels(), fontsize=7, loc="upper left",
                   bbox_to_anchor=(1.02, 1), ncol=1 + len(runs) // 20)
plt.tight_layout(); plt.show()
pd.DataFrame(summary).sort_values("best_acc", ascending=False)

## 7. Download results (if not using Drive)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/ssfl_runs", "zip", RUNS_DIR)
files.download("/content/ssfl_runs.zip")